# 03 — Method C: INT4 **QAT** (양자화 인식 학습)

Method A의 머지 BF16 모델에서 출발해, **양자화를 시뮬레이션(fake-quant)하며 짧게 재학습**한 뒤 B와 **동일한 INT4 포맷**으로 변환한다. 차이는 순수하게 *train-aware* 여부.

> ## ⚙️ 실행 모드 배너 — 이 노트북은 **Azure A100 80GB에서 실제 실행**됨 (프로덕션/스펙 경로)
>
> **컴퓨트:** Azure `Standard_NC24ads_A100_v4` (**NVIDIA A100 80GB PCIe** ×1) · region **japaneast** ·
> `compute.mode: gpu` · **Spot 우선순위**.
>
> **양자화(스펙 C) — QAT (full-param STE):** 입력 = `artifacts/A_bf16/`. tile-packed 서빙 양자화와 **동일한 스킴**의
> fake-quant(`Int4WeightOnlyConfig(g128)`에서 추론)를 base에 삽입 → **양자화되는 linear 가중치만** STE로
> KorQuAD **short fine-tune**(`qat.max_steps=400`, lr=1e-5, 임베딩·헤드 동결)해 가중치가 int4로 양자화된 뒤에도
> 잘 동작하도록 적응 → convert(adapted-bf16) → **B와 동일한 tile-packed INT4**로 export.
> 임베딩·`lm_head` 동결로 held-out ppl 드리프트를 억제하고, 학습은 양자화 대상 가중치의 *적응*에 집중된다.
>
> **재현:** `results/env_C.json`, `results/C_int4_qat_metrics.json`, 3-way 표 C 행.

### 이 방법(C) — 무엇/왜/어떻게
- **무엇:** quantization-aware training(QAT). 학습 중 forward에 int4 양자화를 시뮬레이션(STE로 역전파)해,
  가중치가 *양자화된 뒤에도* 잘 동작하도록 적응시킨다. 여기서는 **양자화 대상 linear 가중치만 full-param STE**로 학습.
- **왜:** PTQ(B)가 노출하는 양자화 오차를 학습으로 보정 → 동일 INT4 포맷에서 품질 회복을 기대.
  임베딩·`lm_head`는 동결해 원 모델의 언어능력(ppl) 드리프트를 억제하고, 학습은 양자화되는 가중치에 집중한다.
- **핵심(스킴 일치):** fake-quant는 `Int4WeightOnlyConfig(g128)`에서 **추론**되어 실제 tile-packed 서빙
  양자화와 **정확히 동일한 int4 스킴**이다. (스킴이 어긋나면 QAT가 엉뚱한 양자화에 적응해 효과가 없다.)
- **어떻게:** ①matched fake-quant 삽입(prepare, `lm_head` 제외) → ②양자화 대상 가중치만 STE로 short fine-tune →
  ③convert로 fake-quant 제거(adapted bf16, 이미 이 int4 스킴에 맞춰 학습됨) → ④B와 **동일한** tile-packed INT4로 export.
  서빙 포맷이 B와 같으므로 B vs C가 곧 *train-aware* 효과.

### 0) 부트스트랩 & 버전 고정 (재현성)
`quantization/` 공용 모듈 import + 버전을 `results/env_C.json`에 기록.

In [1]:
import os, sys
here = os.getcwd()
for cand in [here, os.path.dirname(here), os.path.join(here, "pdf_qa_extraction"),
             os.path.dirname(os.path.dirname(here))]:
    if os.path.isdir(os.path.join(cand, "quantization")):
        if cand not in sys.path:
            sys.path.insert(0, cand)
        os.chdir(cand)
        break
print("cwd:", os.getcwd())

cwd: /home/azureuser/work/pdf_qa_extraction


In [2]:
import json, platform, torch, transformers, torchao
env = {
    "python": platform.python_version(),
    "torch": torch.__version__,
    "transformers": transformers.__version__,
    "torchao": torchao.__version__,
    "cuda_available": torch.cuda.is_available(),
    "device": (torch.cuda.get_device_name(0) if torch.cuda.is_available() else "cpu"),
}
os.makedirs("quantization/results", exist_ok=True)
with open("quantization/results/env_C.json", "w", encoding="utf-8") as fh:
    json.dump(env, fh, ensure_ascii=False, indent=2)
env

{'python': '3.10.12',
 'torch': '2.11.0+cu130',
 'transformers': '5.5.0',
 'torchao': '0.17.0',
 'cuda_available': True,
 'device': 'NVIDIA A100 80GB PCIe'}

### 1) 설정 로드 + A 머지 확인
`config.yaml`의 `qat.*`(group_size·max_steps·lr) 사용. 입력 A 머지 모델 선행 필수.

In [3]:
from quantization.data_korquad import load_config, load_korquad, to_hf_text_dataset
import quantization.eval_qa as E
cfg = load_config()
A_DIR = cfg['paths']['method_a_dir']
C_DIR = cfg['paths']['method_c_dir']
C_BF16 = C_DIR + '_bf16adapt'   # transient adapted-bf16 checkpoint before int4 export
gs = int(cfg['qat']['group_size'])
assert os.path.isdir(A_DIR), f'A 머지 모델 없음: {A_DIR} (먼저 01 실행)'
print('입력 A 머지 :', A_DIR)
print('출력 C(int4):', C_DIR)
print('QAT 설정    :', {k: cfg['qat'][k] for k in ['group_size','max_steps','learning_rate','per_device_batch_size','grad_accum']})

입력 A 머지 : quantization/artifacts/A_bf16
출력 C(int4): quantization/artifacts/C_int4_qat
QAT 설정    : {'group_size': 128, 'max_steps': 400, 'learning_rate': 1e-05, 'per_device_batch_size': 2, 'grad_accum': 4}


### 2) 데이터 — KorQuAD (A/B/C 동일)
QAT fine-tune은 A와 동일한 train 슬라이스를, eval은 동일 held-out을 사용.

In [4]:
data = load_korquad(cfg)
print('train:', len(data['train']), '| eval:', len(data['eval']))

train: 60407 | eval: 500


### 3) QAT (full-param STE) — matched fake-quant → 양자화 가중치 재학습 → convert → INT4 export
① tile-packed 서빙과 **동일 스킴**의 fake-quant 삽입(`lm_head` 제외) → ② **양자화되는 linear 가중치만** STE로 `qat.max_steps` 재학습(임베딩·헤드 동결) → ③ convert로 adapted-bf16 복원 → ④ B와 동일한 tile-packed INT4로 export.

In [5]:
import copy, torch, torch.nn as nn
from transformers import AutoModelForCausalLM, AutoTokenizer, TorchAoConfig
from torchao.quantization import quantize_, Int4WeightOnlyConfig
from torchao.quantization.qat import QATConfig
from torchao.quantization.qat.fake_quantize_config import _infer_fake_quantize_configs
from quantization.train_lora import _build_sft_trainer

def _not_lmhead(m, fqn):
    return isinstance(m, nn.Linear) and 'lm_head' not in fqn

tok = AutoTokenizer.from_pretrained(A_DIR)
if tok.pad_token is None: tok.pad_token = tok.eos_token
model = AutoModelForCausalLM.from_pretrained(A_DIR, dtype=torch.bfloat16, device_map='cuda')

# (1) prepare: fake-quant that MATCHES the tile-packed serving scheme (inferred, not hand-set)
_, wcfg = _infer_fake_quantize_configs(Int4WeightOnlyConfig(group_size=gs))
quantize_(model, QATConfig(weight_config=wcfg, step='prepare'), filter_fn=_not_lmhead)

# (2) train ONLY the fake-quantized linear weights (the int4 targets); freeze embeddings/lm_head/norms
#     -> STE updates the REAL weights so that quant(w) is good after convert (no convert-time gap)
for p in model.parameters(): p.requires_grad_(False)
n_train = 0
for m in model.modules():
    if type(m).__name__ == 'FakeQuantizedLinear':
        m.weight.requires_grad_(True); n_train += m.weight.numel()
n_fq = sum(1 for m in model.modules() if type(m).__name__ == 'FakeQuantizedLinear')
print(f'fake-quant = {type(wcfg).__name__} | FakeQuantizedLinear layers = {n_fq} | trainable = {n_train/1e6:.1f}M (full-param STE, emb/head frozen)')

# (3) short QAT fine-tune on KorQuAD (STE through fake-quant), reuse A's SFT trainer + qat overrides
eos = tok.eos_token or '</s>'
train_ds = to_hf_text_dataset(data['train'], eos)
qcfg = copy.deepcopy(cfg); qcfg['train'] = dict(cfg['train'])
qcfg['train'].update(max_steps=int(cfg['qat']['max_steps']),
                     learning_rate=float(cfg['qat']['learning_rate']),
                     per_device_batch_size=int(cfg['qat']['per_device_batch_size']),
                     grad_accum=int(cfg['qat']['grad_accum']))
trainer = _build_sft_trainer(qcfg, model, tok, train_ds)
trainer.train(); model = trainer.model

# (4) convert: fold fake-quant -> adapted bf16 (weights already tuned FOR this exact int4 scheme)
quantize_(model, QATConfig(step='convert'), filter_fn=_not_lmhead)
os.makedirs(C_BF16, exist_ok=True); model.save_pretrained(C_BF16); tok.save_pretrained(C_BF16)
del model, trainer; torch.cuda.empty_cache()

# (5) export the SAME tile-packed INT4 serving format as B
qmodel = AutoModelForCausalLM.from_pretrained(
    C_BF16, dtype=torch.bfloat16, device_map='cuda',
    quantization_config=TorchAoConfig(quant_type=E.make_int4_weightonly_config(gs)))
os.makedirs(C_DIR, exist_ok=True); qmodel.save_pretrained(C_DIR); tok.save_pretrained(C_DIR)
size_gb = E.dir_size_gb(C_DIR)
print(f'INT4 QAT 저장 완료: {C_DIR}  size={size_gb:.3f} GB')
del qmodel; torch.cuda.empty_cache()

fake-quant = Int4WeightFakeQuantizeConfig | FakeQuantizedLinear layers = 196 | trainable = 1409.3M (full-param STE, emb/head frozen)


Step,Training Loss
10,2.369144
20,2.304231
30,2.226027
40,2.213464
50,2.264223
60,2.138087
70,2.189192
80,2.121254
90,2.125303
100,2.149537


INT4 QAT 저장 완료: quantization/artifacts/C_int4_qat  size=1.288 GB


### 4) 동작 데모 (필수) — held-out 질문 **1개**
저장한 INT4 서빙 아티팩트를 **새로 로드**해 답을 생성.

In [6]:
from transformers import AutoModelForCausalLM
model = AutoModelForCausalLM.from_pretrained(C_DIR, device_map='cuda')
demo = data['eval'][0]
gen = E.generate_answers(model, tok, [demo.prompt], max_new_tokens=32, batch_size=1)
print('[질문]', demo.question)
print('[정답]', demo.answers)
print('[모델 답]', gen['answers'][0])

[질문] 2004년 이명박이 서울시장 재직시절 전면적으로 개선한 것은?
[정답] ['대중교통체계']
[모델 답] 대중교통체계


### 5) 수치 — EM/F1 · perplexity · 크기 · VRAM · tok/s
재로드한 INT4 모델로 A/B와 **동일한 eval** → `results/C_int4_qat_metrics.json` + 3-way 표 C 행 append.

In [7]:
res = E.evaluate_model(model, tok, data['eval'], method='C_int4_qat',
                       base_model=cfg['base_model']['selected'],
                       model_dir=C_DIR,
                       max_new_tokens=cfg['eval']['max_new_tokens'],
                       batch_size=cfg['eval']['batch_size'],
                       ppl_samples=cfg['eval']['ppl_samples'],
                       precision='int4',
                       notes=f"TorchAO int4 QAT full-param g{gs} steps={cfg['qat']['max_steps']} (matched fake-quant, emb/head frozen)")
E.write_metrics(res, cfg['paths']['results_dir'])
E.append_to_table(res, cfg['paths']['results_dir'])
from dataclasses import asdict
row = asdict(res)
print('C (INT4 QAT) — 3-way 표 C 행')
for k in ['method','base_model','exact_match','f1','perplexity','size_gb','peak_vram_gb','tok_per_s','precision']:
    print(f'  {k:14}: {row[k]}')

C (INT4 QAT) — 3-way 표 C 행
  method        : C_int4_qat
  base_model    : Qwen/Qwen3-1.7B
  exact_match   : 71.8
  f1            : 83.521
  perplexity    : 12.9728
  size_gb       : 1.2878
  peak_vram_gb  : 7.7992
  tok_per_s     : 36.99
  precision     : int4


### 6) 3-way 비교 & 다음 단계
이로써 `three_way_table.json`에 A(BF16 LoRA)·B(INT4 PTQ)·C(INT4 QAT) 3행이 모두 채워진다. B와 C는 동일한 tile-packed INT4 서빙 포맷이므로 크기·VRAM은 유사하고, EM/F1·ppl 차이가 **QAT의 품질 회복 효과**를 보여준다. 다음: vLLM INT4 서빙 벤치(선택) + README 결과표 갱신.